# w9_cv_test_tag.ipynb — 回溯补算 test_tag(纯评测,零重训)

`test_tag` = 干净 inductive tag F1(探针只在 1,613 训练池上拟合,阈值选
在 val_g,评测 test_g 的 neutral 查询)。它是 post-hoc:塔投影 npz 已存
SPg/SPa/SPq,worker 在 DONE_FLAG 存在时跳过训练、直奔轨迹刷新,刷新块用
zs_from_arrays(现含 test_tag)在 CPU 上重算并重写 zsbest。本 notebook 对
论文每个臂 x 5 折跑一遍,产出带 test_tag 的 zsbest,并打印五折汇总表。
基线五臂 + swin/MoCo/Two-stage(bw)+ CE/I-CE 参照。AUTO-STOPS。


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_cv_out"       # CV campaign out dir

RECIPES = ["wcle_slot8i2cemean_icetf",   # 8-slot i2ce: capacity judge @4096
           "wcle_mq3072i2ce_icetf",      # I2CE (MoCo q3072)
           "wcle_mq3072ce_cetf",         # CE (MoCo q3072)
           "wcle_byol_bytf",             # BYOL baseline
           "wcle_epd_v20i10c20_cetf",    # VICReg epd 20/10/20 (bs=192)
           "wcle_bce_cetf"]              # SimCLR-style in-batch NT-Xent
REF_RECIPES = ["wcle_ce_cetf", "wcle_i2ce_icetf"]   # wave-1 rows (read-only)
CAPS = [4096]
N_FOLDS = 5
EPOCHS, CKPT_EVERY, CKPT_SEEDS, TOPUP_SEEDS = 2000, 50, 2, 10   # ZS: seeds unused

# VRAM scheduler knobs
SAFETY = 0.85
RESERVE_GIB = 1.5

# 48G cards (L40) can't fit the 4096 grad-gallery cells (~45G): pods whose
# smallest GPU has <60GiB free skip caps above MAX_CAP_48G, run their share
# (512/1024/2048 = 30 towers) and AUTO-STOP early -- no waiting on the
# A100 pod, no OOM-burned claims. 80G pods run all 40.
MAX_CAP_48G = 2048

def nm_of(r, k, cap):
    return f"w9cv_{r}_fold{k}" + (f"_g{cap}" if cap != 512 else "")

os.makedirs(OUT_DIR, exist_ok=True)
print("jobs:", len(RECIPES) * N_FOLDS * len(CAPS),
      f"({len(RECIPES)} recipes x {N_FOLDS} folds x {len(CAPS)} caps) @ {EPOCHS}ep")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (same file set as w9_a100.ipynb).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz", "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)

In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# EVAL-ONLY backfill: --eval-only makes the worker skip training entirely
# and just refresh traj + zsbest from the existing tower npz (epoch-agnostic,
# so it does not matter which ep each arm trained to). test_tag/val_tag are
# recomputed on CPU from the stored SPg/SPa/SPq and best is RE-SELECTED by the
# new cvsel = noname_h1 + noname_h5 + 2*val_tag. Idempotent + resume-safe.
import json, os, subprocess, time
from pathlib import Path

CAP, N_FOLDS = 4096, 5
PAPER_ARMS = [
    "wcle_i2ce_icetf", "wcle_ce_cetf", "wcle_bce_cetf",
    "wcle_epd_v20i10c20_cetf", "wcle_byol_bytf",
    "wcle_swin168step84loop2i2ce_icetf", "wcle_mq3072i2ce_icetf",
]
gpus = J.detect_gpus()
logd = Path(OUT_DIR) / "logs"; logd.mkdir(parents=True, exist_ok=True)
fails = []

def _worker_cmd(r, k, extra=()):
    return ["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
            OUT_DIR, "--repo", REPO, "--arm", r, "--fold", str(k),
            "--n-folds", str(N_FOLDS), "--anchor-cap", str(CAP), "--epochs", "1",
            "--eval-only", "--full-pool", "--full-pool-path", FULL_POOL_PATH
            ] + list(extra)

def _run(cmd, log):
    with open(logd / log, "w") as fh:
        return subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                              env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpus[0],
                                       PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True"))

def run_eval(r, k, extra=(), sfx=""):
    nm = f"w9cv_{r}_fold{k}_g{CAP}{sfx}"
    towers = list(Path(OUT_DIR).glob(f"tower_{nm}_fp_ep*.npz"))
    zb = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
    if not towers:
        print(f"[missing] {nm}: no tower npz on the volume -- skip", flush=True); return
    if zb.exists() and "val_tag" in json.loads(zb.read_text()):
        print(f"[has] {nm}: already re-selected (val_tag present)", flush=True); return
    print(f"[eval] {nm} ({len(towers)} towers) ...", flush=True)
    t0 = time.time()
    rc = _run(_worker_cmd(r, k, extra), f"testtag_{nm}.log")
    ok = zb.exists() and "val_tag" in json.loads(zb.read_text())
    if not ok or rc.returncode != 0:
        fails.append(nm)
    print(f"[eval] {nm}: {'OK' if ok else 'NO val_tag'} rc={rc.returncode} "
          f"[{time.time()-t0:.0f}s]", flush=True)

for r in PAPER_ARMS:
    for k in range(N_FOLDS):
        run_eval(r, k)
# Two-stage concat: _bw towers if present (--init-ckpt only reproduces the
# _bw name; no file is read under --eval-only).
for k in range(N_FOLDS):
    run_eval("wcle_swin168step84loop2i2ce_icetf", k,
             extra=("--init-ckpt", "eval-only-placeholder"), sfx="_bw")

# FROZEN baseline (metrics4 path): per fold, only recompute when test_tag is
# missing -- delete just that fold's json and force ONE worker run for it.
# Never deletes a frozen json that already carries test_tag (resume-safe).
FROZEN_ARM = "wcle_i2ce_icetf"
for k in range(N_FOLDS):
    fj = Path(OUT_DIR) / f"w9cv_frozen_fold{k}.json"
    if fj.exists() and "test_tag" in json.loads(fj.read_text()):
        print(f"[frozen] fold{k}: has test_tag", flush=True); continue
    tw = list(Path(OUT_DIR).glob(
        f"tower_w9cv_{FROZEN_ARM}_fold{k}_g{CAP}_fp_ep*.npz"))
    if not tw:
        print(f"[frozen] fold{k}: no {FROZEN_ARM} tower -- skip", flush=True); continue
    fj.unlink(missing_ok=True)   # absent -> the worker's metrics4 recomputes it
    _run(_worker_cmd(FROZEN_ARM, k), f"frozen_fold{k}.log")
    ok = fj.exists() and "test_tag" in json.loads(fj.read_text())
    if not ok:
        fails.append(f"frozen_fold{k}")
    print(f"[frozen] fold{k}: {'OK' if ok else 'FAIL'}", flush=True)

print(f"eval-only pass complete; {len(fails)} failure(s): {fails}", flush=True)


In [ ]:
# Five-fold test_tag table (vs the old tag_neutral / tag_noname columns).
import json
import numpy as np
from pathlib import Path

def rows(arm, sfx=""):
    out = {}
    for k in range(N_FOLDS):
        p = Path(OUT_DIR) / f"zsbest_w9cv_{arm}_fold{k}_g{CAP}{sfx}_fp.json"
        if p.exists():
            out[k] = json.loads(p.read_text())
    return out

def line(lab, rws):
    if not rws:
        print(f"{lab:26s} (none)"); return
    def ms(f):
        v = [r[f] for r in rws.values() if f in r]
        return (np.mean(v), np.std(v)) if v else (float("nan"), float("nan"))
    tt = ms("test_tag"); tn = ms("tag_neutral"); tx = ms("tag_noname")
    print(f"{lab:26s} n={len(rws)}  test_tag {tt[0]:.3f}±{tt[1]:.3f}  "
          f"(neu {tn[0]:.3f} / noname {tx[0]:.3f})")

for arm, lab in (("wcle_i2ce_icetf", "I-CE (ours)"),
                 ("wcle_swin168step84loop2i2ce_icetf", "swin-I-CE"),
                 ("wcle_ce_cetf", "CE only"),
                 ("wcle_bce_cetf", "SimCLR-style"),
                 ("wcle_epd_v20i10c20_cetf", "VICReg"),
                 ("wcle_byol_bytf", "BYOL"),
                 ("wcle_mq3072i2ce_icetf", "MoCo-I2CE")):
    line(lab, rows(arm))
line("Two-stage (bw)", rows("wcle_swin168step84loop2i2ce_icetf", "_bw"))

# frozen baseline (metrics4 path): test_tag lives at the top level
fz = {}
for k in range(N_FOLDS):
    p = Path(OUT_DIR) / f"w9cv_frozen_fold{k}.json"
    if p.exists():
        d = json.loads(p.read_text())
        if "test_tag" in d:
            fz[k] = d
if fz:
    tt = [d["test_tag"] for d in fz.values()]
    on = [d["noname"]["tag"] for d in fz.values() if "noname" in d]
    print(f"{'Frozen embedder':26s} n={len(fz)}  test_tag "
          f"{np.mean(tt):.3f}±{np.std(tt):.3f}  (noname {np.mean(on):.3f})")


In [ ]:
# AUTO-STOP: stop THIS pod when the queue has finished (results live on the
# network volume; idle GPU time is pure waste). Uses the hardened ladder in
# VICReg_review/pod_selfstop.py. Set AUTO_STOP=False to keep the pod alive.
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    fails = globals().get("fails", [])
    if fails:
        print(f"NOTE: {len(fails)} job(s) FAILED -- logs in {OUT_DIR}/logs; "
              "stopping anyway to avoid idle burn.")
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- remember to stop the pod yourself.")